# 13_bi_views
Creates the extra views that Power BI reads (audit / error-logging views and the top-game CCU trend view). Safe to re-run; the views are always current, so nothing needs refreshing after each pipeline run.

Run `09_gold_setup` first (it creates `vw_fact_latest` and the other `gold_*` views).

In [ ]:
spark.sql("USE workspace.steam")

## 1. Audit views (execution log, quarantine, schema drift)

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW audit_pipeline_runs AS
SELECT run_id,
       layer,
       CASE layer WHEN 'Raw-to-Bronze' THEN 1 WHEN 'Bronze-to-Silver' THEN 2
                  WHEN 'Silver-to-Gold' THEN 3 ELSE 4 END                         AS layer_order,
       CASE WHEN layer = 'Silver-to-Gold' THEN split(file_processed, ' ')[0] END  AS gold_table,
       load_type,
       file_processed,
       status,
       to_date(start_time)                                                        AS run_date,
       start_time,
       end_time,
       ROUND((unix_millis(end_time) - unix_millis(start_time)) / 1000.0, 2)       AS duration_sec,
       COALESCE(rows_inserted, 0)         AS rows_inserted,
       COALESCE(rows_updated, 0)          AS rows_updated,
       COALESCE(rows_skipped_unchanged,0) AS rows_skipped_unchanged,
       COALESCE(rows_quarantined, 0)      AS rows_quarantined,
       error_message
FROM pipeline_execution_logs
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW audit_quarantine_reasons AS
SELECT load_date, trim(r) AS reject_reason, COUNT(*) AS quarantined_rows
FROM silver_quarantine
LATERAL VIEW explode(split(reject_reason, ';')) x AS r
GROUP BY load_date, trim(r)
""")

spark.sql("""
CREATE OR REPLACE VIEW audit_schema_drift AS
SELECT column_name, drift_type, detected_on, source_file, load_timestamp
FROM schema_drift_log
""")

## 2. Trend view
Only the 25 games with the most concurrent players. Do **not** plot `SUM(ccu)` per date over all games: the first date is a full load (about 82,000 rows) and later dates hold only the games that changed, so the total would look like a collapse.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold_daily_ccu_trends AS
SELECT d.full_date, f.date_key, g.game_key, g.name AS game_name,
       f.ccu, f.total_reviews, f.approval_rate, f.price_usd
FROM fact_game_daily f
JOIN dim_game g ON f.game_key = g.game_key
JOIN dim_date d ON f.date_key = d.date_key
WHERE f.game_key IN (SELECT game_key FROM vw_fact_latest ORDER BY ccu DESC LIMIT 25)
""")

## 3. Quick check

In [ ]:
for v in ["audit_pipeline_runs", "audit_quarantine_reasons", "audit_schema_drift", "gold_daily_ccu_trends"]:
    print(v, spark.table(v).count(), "rows")

display(spark.sql("""
    SELECT run_date, layer, gold_table, status, duration_sec, rows_inserted, rows_updated, rows_skipped_unchanged
    FROM audit_pipeline_runs ORDER BY start_time DESC LIMIT 12
"""))